In [1]:
import sys
print(sys.executable)

e:\LandSlide-Prediction\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import zipfile

import geopandas as gpd
import pyogrio
import pandas as pd

from shapely.geometry import box
from shapely import make_valid

In [3]:
PROJECT_DIR = Path("..").resolve()

RAW_DIR = PROJECT_DIR / "data" / "raw"
COORD_DIR = PROJECT_DIR / "data" / "coordinates"

COORD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

KMZ_PATH = RAW_DIR / "Lanslides_Ditwa_2025.kmz"

print(PROJECT_DIR)
print(KMZ_PATH)

E:\LandSlide-Prediction
E:\LandSlide-Prediction\data\raw\Lanslides_Ditwa_2025.kmz


In [4]:
EXTRACT_DIR = RAW_DIR / "kmz_extracted"

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with zipfile.ZipFile(KMZ_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

kml_files = list(
    EXTRACT_DIR.rglob("*.kml")
)

print(kml_files)

[WindowsPath('E:/LandSlide-Prediction/data/raw/kmz_extracted/doc.kml')]


In [5]:
kml_path = kml_files[0]

layers = pyogrio.list_layers(
    kml_path
)

print(layers)

[['Lanslides_Ditwa_2025' 'Unknown']]


In [6]:
layer_name = layers[0][0]

gdf = gpd.read_file(
    kml_path,
    layer=layer_name,
    engine="pyogrio"
)

print("Rows:", len(gdf))
print("CRS:", gdf.crs)

print(
    gdf.geometry.geom_type.value_counts()
)

Rows: 4225
CRS: EPSG:4326
MultiPolygon    4225
Name: count, dtype: int64


In [7]:
gdf = gdf[
    gdf.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].copy()

gdf = gdf[
    gdf.geometry.notna()
].copy()

gdf = gdf[
    ~gdf.geometry.is_empty
].copy()

In [8]:
if gdf.crs is None:

    gdf = gdf.set_crs(
        "EPSG:4326"
    )

else:

    gdf = gdf.to_crs(
        "EPSG:4326"
    )

print(gdf.crs)

EPSG:4326


In [9]:
gdf["geometry"] = gdf.geometry.apply(
    lambda geom:
        make_valid(geom)
        if not geom.is_valid
        else geom
)

print(
    "Invalid:",
    (~gdf.geometry.is_valid).sum()
)

Invalid: 0


In [10]:
gdf = gdf.reset_index(
    drop=True
)

gdf["landslide_id"] = [
    f"LS_{i:05d}"
    for i in range(
        1,
        len(gdf) + 1
    )
]

In [11]:
gdf["sample_point"] = (
    gdf.geometry.representative_point()
)

gdf["longitude"] = (
    gdf["sample_point"].x
)

gdf["latitude"] = (
    gdf["sample_point"].y
)

In [12]:
gdf_utm = gdf.to_crs(
    "EPSG:32644"
)

gdf["area_m2"] = (
    gdf_utm.geometry.area.values
)

gdf["perimeter_m"] = (
    gdf_utm.geometry.length.values
)

In [13]:
PATCH_PIXELS = 128
RESOLUTION_M = 10

PATCH_SIZE_M = (
    PATCH_PIXELS
    * RESOLUTION_M
)

HALF_SIZE_M = (
    PATCH_SIZE_M / 2
)

In [14]:
points_utm = (
    gdf_utm.geometry
    .representative_point()
)

boxes = []

for point in points_utm:

    boxes.append(
        box(
            point.x - HALF_SIZE_M,
            point.y - HALF_SIZE_M,
            point.x + HALF_SIZE_M,
            point.y + HALF_SIZE_M
        )
    )

In [15]:
patch_gdf = gpd.GeoDataFrame(
    {
        "landslide_id":
            gdf["landslide_id"]
    },
    geometry=boxes,
    crs="EPSG:32644"
)

patch_gdf = patch_gdf.to_crs(
    "EPSG:4326"
)

In [16]:
bounds = patch_gdf.geometry.bounds

gdf["min_lon"] = bounds.minx.values
gdf["min_lat"] = bounds.miny.values

gdf["max_lon"] = bounds.maxx.values
gdf["max_lat"] = bounds.maxy.values

In [17]:
coordinates = gdf[
    [
        "landslide_id",
        "longitude",
        "latitude",
        "area_m2",
        "perimeter_m",
        "min_lon",
        "min_lat",
        "max_lon",
        "max_lat"
    ]
].copy()

csv_path = (
    COORD_DIR /
    "landslide_coordinates.csv"
)

coordinates.to_csv(
    csv_path,
    index=False
)

print("Saved:", csv_path)

Saved: E:\LandSlide-Prediction\data\coordinates\landslide_coordinates.csv


In [18]:
import geopandas as gpd
import pyogrio
from pathlib import Path

# ------------------------------------------------
# 1. KML path
# ------------------------------------------------

kml_path = Path(
    "../data/raw/kmz_extracted/doc.kml"
)

# ------------------------------------------------
# 2. Check layers
# ------------------------------------------------

layers = pyogrio.list_layers(kml_path)

print("Layers:")
print(layers)

# Your layer was:
# Lanslides_Ditwa_2025

layer_name = layers[0][0]

# ------------------------------------------------
# 3. Read KML
# ------------------------------------------------

gdf = gpd.read_file(
    kml_path,
    layer=layer_name,
    engine="pyogrio"
)

print("\nTotal features:", len(gdf))
print("CRS:", gdf.crs)

print("\nGeometry types:")
print(gdf.geometry.geom_type.value_counts())

Layers:
[['Lanslides_Ditwa_2025' 'Unknown']]

Total features: 4225
CRS: EPSG:4326

Geometry types:
MultiPolygon    4225
Name: count, dtype: int64


In [19]:
print(gdf.geometry.geom_type.value_counts())

MultiPolygon    4225
Name: count, dtype: int64
